# Polygon.io — Data Ingestion

Pulls **2 years** of daily bars for stocks, options, and futures.  
Rate-limited to **5 req/min** (free tier) with automatic retry (exponential back-off) and **checkpoint/resume** (existing parquet files are skipped).

```
data/raw/
  stocks/           {TICKER}_day.parquet
  options/
    snapshots/      {UNDERLYING}_{DATE}.parquet   <- contract reference (ticker, strike, expiry)
    history/        {CONTRACT}_day.parquet        <- OHLCV
  futures/          {CONTRACT}_day.parquet
```

> **Note:** Greeks, IV, and open interest require the Starter plan ($29/mo) and are not fetched here.

In [1]:
import os, subprocess, sys
from pathlib import Path
from dotenv import load_dotenv

# Running under VS Code (local kernel) inherits VSCODE_* env vars — treat as a
# local dev session even if /content happens to exist, so we never clobber the
# repo file with a redundant clone/Drive mount.
IN_VSCODE = bool(os.environ.get('VSCODE_PID') or os.environ.get('VSCODE_CWD'))
IN_COLAB  = ('google.colab' in sys.modules or os.path.exists('/content')) and not IN_VSCODE

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    _token = userdata.get('GITHUB_TOKEN')
    _repo  = 'shreyasnat2804/JEPA-quant'
    _dest  = '/content/JEPA-quant'

    # Pull the latest repo code (.py modules, etc.) into the VM.
    # NOTE: this does NOT refresh the notebook you're currently viewing — a cell
    # can't reload its own tab. To get the latest notebook, reopen it via
    # File -> Open notebook -> GitHub tab (or Runtime -> revert if editing in Colab).
    if not os.path.exists(_dest):
        subprocess.run(
            ['git', 'clone', f'https://{_token}@github.com/{_repo}.git', _dest],
            check=True,
        )
    else:
        subprocess.run(['git', '-C', _dest, 'pull'], check=True)

    os.chdir(f'{_dest}/notebooks')
    print(f'Repo synced. Working directory: {os.getcwd()}')
else:
    load_dotenv('../.env')
    where = 'VS Code' if IN_VSCODE else 'local'
    print(f'Running in {where} — skipping Drive mount and git pull')


Running in VS Code — skipping Drive mount and git pull


In [2]:
%pip install aiohttp nest-asyncio python-dotenv tqdm pandas pyarrow --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import asyncio
import logging
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Optional

import aiohttp
import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

nest_asyncio.apply()  # allow asyncio.run() inside Jupyter

# Colab: falls back to Colab Secrets (add MARKET_DATA_API_KEY in the 🔑 tab)
# Local: already loaded from .env in the setup cell above
load_dotenv('../.env')

if not os.environ.get('MARKET_DATA_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['MARKET_DATA_API_KEY'] = userdata.get('MARKET_DATA_API_KEY')
    except Exception:
        raise EnvironmentError(
            'MARKET_DATA_API_KEY not found.\n'
            '  Local: add it to JEPA-quant/.env\n'
            '  Colab: Secrets tab (🔑) → add MARKET_DATA_API_KEY'
        )

API_KEY    = os.environ['MARKET_DATA_API_KEY']
BASE_URL   = 'https://api.polygon.io'
RATE_LIMIT = 0.08  # 5 requests/minute (free tier)

END_DATE   = date.today().isoformat()
START_DATE = (date.today() - timedelta(days=730)).isoformat()

# Colab writes to Drive; local runs write to data/raw/ in the project root
if IN_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw')
else:
    DATA_DIR = Path('../data/raw')

for sub in ('stocks', 'options/snapshots', 'options/history', 'futures'):
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('polygon_ingest')

print(f'Date range: {START_DATE}  ->  {END_DATE}')
print(f'Data directory: {DATA_DIR}')

Date range: 2024-06-02  ->  2026-06-02
Data directory: ../data/raw


/Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class TokenBucket:
    """Async token-bucket rate limiter."""

    def __init__(self, rate: float):
        self.rate = rate
        self.tokens = float(rate)
        self._last = time.monotonic()
        self._lock = asyncio.Lock()

    async def acquire(self) -> None:
        async with self._lock:
            now = time.monotonic()
            self.tokens = min(self.rate, self.tokens + (now - self._last) * self.rate)
            self._last = now
            if self.tokens < 1.0:
                await asyncio.sleep((1.0 - self.tokens) / self.rate)
                self.tokens = 0.0
            else:
                self.tokens -= 1.0

In [ ]:
import ssl
import certifi


class PolygonClient:
    _MAX_RETRIES = 5

    def __init__(self, api_key: str, rate: float = 5.0):
        self.api_key = api_key
        self._limiter = TokenBucket(rate)
        self._session: Optional[aiohttp.ClientSession] = None
        # macOS python.org builds don't trust the system keychain; point aiohttp
        # at certifi's CA bundle so TLS verification works everywhere (incl. Colab).
        self._ssl_ctx = ssl.create_default_context(cafile=certifi.where())

    async def _get_session(self) -> aiohttp.ClientSession:
        if self._session is None or self._session.closed:
            self._session = aiohttp.ClientSession(
                headers={'Authorization': f'Bearer {self.api_key}'},
                timeout=aiohttp.ClientTimeout(total=30),
                connector=aiohttp.TCPConnector(ssl=self._ssl_ctx),
            )
        return self._session

    async def close(self) -> None:
        if self._session and not self._session.closed:
            await self._session.close()

    async def get(self, url: str, params: Optional[dict] = None) -> dict:
        session = await self._get_session()
        for attempt in range(self._MAX_RETRIES):
            await self._limiter.acquire()
            try:
                async with session.get(url, params=params) as resp:
                    if resp.status == 429:
                        # Sleep out the full minute window — exponential backoff
                        # wastes requests retrying before the limit resets.
                        log.warning('429 rate-limit; sleeping 61s')
                        await asyncio.sleep(61)
                        continue
                    resp.raise_for_status()
                    return await resp.json()
            except (aiohttp.ClientError, asyncio.TimeoutError) as exc:
                if attempt == self._MAX_RETRIES - 1:
                    raise
                wait = 2 ** attempt
                log.warning('Error (attempt %d): %s -- retry in %ds', attempt + 1, exc, wait)
                await asyncio.sleep(wait)
        return {}

    async def paginate(self, url: str, params: Optional[dict] = None) -> list:
        """Follow Polygon next_url cursors to collect all pages."""
        all_results = []
        first = True
        while url:
            data = await self.get(url, params if first else None)
            first = False
            all_results.extend(data.get('results') or [])
            url = data.get('next_url', '')
        return all_results


client = PolygonClient(API_KEY, RATE_LIMIT)
print('Client ready.')

## 1 - Stocks

One paginated request per ticker — 2 years of daily bars fit in a single call (Polygon returns up to 50 000 bars per page).  
Existing `.parquet` files are skipped on re-runs.

In [6]:
STOCK_TICKERS = [
    # Core large-cap universe -- edit to match your strategy
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA',
    'JPM',  'V',    'MA',   'BAC',  'GS',    'MS',
    'XOM',  'CVX',  'COP',
    'LLY',  'UNH',  'PFE',  'MRK',  'ABBV',
    'AVGO', 'AMD',  'INTC', 'QCOM', 'TXN',
    'HD',   'WMT',  'COST', 'TGT',
    'CAT',  'RTX',  'HON',  'DE',
    'NEE',  'DUK',  'SO',
    # Broad ETFs
    'SPY',  'QQQ',  'IWM',  'DIA',  'GLD',  'SLV',  'USO',  'TLT',
]
STOCK_TICKERS = sorted(set(STOCK_TICKERS))
print(f'{len(STOCK_TICKERS)} tickers in universe')

45 tickers in universe


In [7]:
_OHLCV_COLS = ['ticker', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'transactions']


def _empty_ohlcv(ticker: str, path: Path) -> pd.DataFrame:
    df = pd.DataFrame(columns=_OHLCV_COLS)
    df.index.name = 'ts'
    df.to_parquet(path)
    return df


async def fetch_stock_bars(
    ticker: str,
    timespan: str = 'day',
    multiplier: int = 1,
) -> pd.DataFrame:
    out = DATA_DIR / 'stocks' / f'{ticker}_{timespan}.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{ticker}/range'
        f'/{multiplier}/{timespan}/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'true', 'sort': 'asc'})
    if not rows:
        log.warning('No bars: %s — caching empty result', ticker)
        return _empty_ohlcv(ticker, out)

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = ticker
    df = df.set_index('ts')[_OHLCV_COLS]
    df.to_parquet(out)
    return df


async def ingest_stocks(timespans=None):
    timespans = timespans or ['day']
    pairs = [(t, ts) for t in STOCK_TICKERS for ts in timespans]

    def _path(t, ts): return DATA_DIR / 'stocks' / f'{t}_{ts}.parquet'

    cached  = [(t, ts) for t, ts in pairs if     _path(t, ts).exists()]
    pending = [(t, ts) for t, ts in pairs if not _path(t, ts).exists()]

    results = {f'{t}_{ts}': pd.read_parquet(_path(t, ts)) for t, ts in cached}

    with tqdm(total=len(pairs), initial=len(cached), desc='Stocks') as pbar:
        for ticker, timespan in pending:
            results[f'{ticker}_{timespan}'] = await fetch_stock_bars(ticker, timespan)
            pbar.update(1)

    return results

In [ ]:
stock_data = asyncio.run(ingest_stocks(['day']))

ok = {k: v for k, v in stock_data.items() if not v.empty}
print(f'\nFetched {len(ok)}/{len(stock_data)} tickers')
if ok:
    k0 = next(iter(ok))
    print(f'\nSample -- {k0}:')
    print(ok[k0].tail(3).to_string())

Stocks:   0%|          | 0/45 [00:00<?, ?it/s]

## 2 - Options

Uses the **reference endpoint** (`/v3/reference/options/contracts`) to discover contracts filtered by DTE and moneyness, then fetches 2 years of daily OHLCV via `/v2/aggs`.

> Greeks, IV, and open interest are not available on the free tier.  
> Filters: DTE 0–90, strike ±20% of spot.

In [ ]:
OPTIONS_UNDERLYINGS = ['AAPL', 'MSFT', 'NVDA', 'SPY', 'QQQ', 'TSLA', 'AMZN', 'GOOGL']

OPT_MIN_DTE    = 0    # minimum days to expiry
OPT_MAX_DTE    = 90   # maximum days to expiry
OPT_STRIKE_PCT = 20   # +/- % of underlying spot price

print(f'Underlyings: {OPTIONS_UNDERLYINGS}')
print(f'Filters: DTE {OPT_MIN_DTE}-{OPT_MAX_DTE}, strike +/-{OPT_STRIKE_PCT}% spot')

In [ ]:
async def _fetch_prev_close(ticker: str) -> float:
    data = await client.get(f'{BASE_URL}/v2/aggs/ticker/{ticker}/prev')
    results = data.get('results') or [{}]
    return float(results[0].get('c', 0)) if results else 0.0


async def fetch_options_contracts(underlying: str, spot: float) -> pd.DataFrame:
    today = date.today().isoformat()
    out = DATA_DIR / 'options' / 'snapshots' / f'{underlying}_{today}.parquet'
    if out.exists():
        return pd.read_parquet(out)

    exp_max = (date.today() + timedelta(days=OPT_MAX_DTE)).isoformat()

    url = f'{BASE_URL}/v3/reference/options/contracts'
    params = {
        'underlying_ticker': underlying,
        'expiration_date.gte': today,
        'expiration_date.lte': exp_max,
        'strike_price.gte': round(spot * (1 - OPT_STRIKE_PCT / 100), 2),
        'strike_price.lte': round(spot * (1 + OPT_STRIKE_PCT / 100), 2),
        'limit': 250,
    }
    rows = await client.paginate(url, params)

    df = pd.DataFrame(rows) if rows else pd.DataFrame()
    if not rows:
        log.warning('No contracts: %s — caching empty result', underlying)
    else:
        df['underlying'] = underlying
    df.to_parquet(out, index=False)
    return df


async def ingest_options_contracts() -> dict:
    spots = {und: await _fetch_prev_close(und) for und in OPTIONS_UNDERLYINGS}
    contracts = {}
    for und in tqdm(OPTIONS_UNDERLYINGS, desc='Options contracts'):
        spot = spots.get(und, 0)
        if spot > 0:
            contracts[und] = await fetch_options_contracts(und, spot)
        else:
            log.warning('No spot price for %s — skipping', und)
            contracts[und] = pd.DataFrame()
    return contracts

In [ ]:
contracts_data = asyncio.run(ingest_options_contracts())

ok_contracts = {k: v for k, v in contracts_data.items() if not v.empty}
print(f'\nContracts: {len(ok_contracts)}/{len(OPTIONS_UNDERLYINGS)} underlyings')
for und, df in ok_contracts.items():
    print(f'  {und}: {len(df):,} contracts')

In [ ]:
async def fetch_options_bar(contract_ticker: str) -> pd.DataFrame:
    safe = contract_ticker.replace(':', '_').replace('/', '_')
    out = DATA_DIR / 'options' / 'history' / f'{safe}_day.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{contract_ticker}/range'
        f'/1/day/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'false', 'sort': 'asc'})
    if not rows:
        return _empty_ohlcv(contract_ticker, out)

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = contract_ticker
    df = df.set_index('ts')[_OHLCV_COLS]
    df.to_parquet(out)
    return df


async def ingest_options_history(contracts_data: dict) -> dict:
    all_tickers = []
    for df in contracts_data.values():
        if not df.empty and 'ticker' in df.columns:
            all_tickers.extend(df['ticker'].dropna().tolist())
    all_tickers = list(set(all_tickers))

    def _path(t):
        safe = t.replace(':', '_').replace('/', '_')
        return DATA_DIR / 'options' / 'history' / f'{safe}_day.parquet'

    pending  = [t for t in all_tickers if not _path(t).exists()]
    n_cached = len(all_tickers) - len(pending)
    log.info('%d contracts total — %d cached, %d to fetch', len(all_tickers), n_cached, len(pending))

    history = {}
    with tqdm(total=len(all_tickers), initial=n_cached, desc='Options history') as pbar:
        for ticker in pending:
            history[ticker] = await fetch_options_bar(ticker)
            pbar.update(1)

    return history

In [ ]:
options_history = asyncio.run(ingest_options_history(contracts_data))

ok_hist = {k: v for k, v in options_history.items() if not v.empty}
total_bars = sum(len(v) for v in ok_hist.values())
print(f'\nHistory: {len(ok_hist)}/{len(options_history)} contracts  |  {total_bars:,} total bars')

## 3 - Futures

Polygon continuous-contract tickers use the `{ROOT}1!` convention (front month).  
**Verify your exact ticker symbols** against the Polygon/Massive reference API before running — the format can differ from the examples below.
Use `/v3/reference/tickers?market=futures` or the Massive dashboard to confirm.

In [ ]:
FUTURES_TICKERS = [
    # Equity index
    'ES1!',   # E-mini S&P 500
    'NQ1!',   # E-mini Nasdaq-100
    'RTY1!',  # E-mini Russell 2000
    'YM1!',   # E-mini Dow Jones
    # Energy
    'CL1!',   # WTI Crude Oil
    'NG1!',   # Natural Gas
    # Metals
    'GC1!',   # Gold
    'SI1!',   # Silver
    'HG1!',   # Copper
    # Rates
    'ZB1!',   # 30-Year T-Bond
    'ZN1!',   # 10-Year T-Note
    # Grains
    'ZC1!',   # Corn
    'ZS1!',   # Soybeans
    'ZW1!',   # Wheat
]
print(f'{len(FUTURES_TICKERS)} futures contracts configured')

In [ ]:
async def fetch_futures_bars(
    ticker: str,
    timespan: str = 'day',
    multiplier: int = 1,
) -> pd.DataFrame:
    safe = ticker.replace('!', 'cont').replace(':', '_').replace('/', '_')
    out = DATA_DIR / 'futures' / f'{safe}_{timespan}.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{ticker}/range'
        f'/{multiplier}/{timespan}/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'true', 'sort': 'asc'})
    if not rows:
        log.warning('No data: %s — caching empty result', ticker)
        return _empty_ohlcv(ticker, out)

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = ticker
    df = df.set_index('ts')[_OHLCV_COLS]
    df.to_parquet(out)
    return df


async def ingest_futures(timespans=None):
    timespans = timespans or ['day']
    pairs = [(t, ts) for t in FUTURES_TICKERS for ts in timespans]

    def _path(t, ts):
        safe = t.replace('!', 'cont').replace(':', '_').replace('/', '_')
        return DATA_DIR / 'futures' / f'{safe}_{ts}.parquet'

    cached  = [(t, ts) for t, ts in pairs if     _path(t, ts).exists()]
    pending = [(t, ts) for t, ts in pairs if not _path(t, ts).exists()]

    results = {f'{t}_{ts}': pd.read_parquet(_path(t, ts)) for t, ts in cached}

    with tqdm(total=len(pairs), initial=len(cached), desc='Futures') as pbar:
        for ticker, timespan in pending:
            results[f'{ticker}_{timespan}'] = await fetch_futures_bars(ticker, timespan)
            pbar.update(1)

    return results

In [ ]:
futures_data = asyncio.run(ingest_futures(['day']))

ok_fut = {k: v for k, v in futures_data.items() if not v.empty}
print(f'\nFetched {len(ok_fut)}/{len(futures_data)} futures')
if ok_fut:
    k0 = next(iter(ok_fut))
    print(f'\nSample -- {k0}:')
    print(ok_fut[k0].tail(3).to_string())

## 4 - Validation

In [ ]:
print('=' * 62)
print('INGESTION SUMMARY')
print('=' * 62)

stock_ok = {k: v for k, v in stock_data.items()     if not v.empty}
chain_ok = {k: v for k, v in contracts_data.items() if not v.empty}
fut_ok   = {k: v for k, v in futures_data.items()   if not v.empty}

# Options history: count files on disk — dict only holds newly-fetched contracts
hist_files   = list((DATA_DIR / 'options' / 'history').glob('*.parquet'))
n_hist       = len(hist_files)
total_hist_bars = sum(len(pd.read_parquet(f)) for f in hist_files)

print(f'\nStocks:            {len(stock_ok):>4} tickers      {sum(len(v) for v in stock_ok.values()):>10,} bars')
print(f'Options contracts: {len(chain_ok):>4} underlyings  {sum(len(v) for v in chain_ok.values()):>10,} contracts')
print(f'Options history:   {n_hist:>4} contracts    {total_hist_bars:>10,} bars  (on disk)')
print(f'Futures:           {len(fut_ok):>4} contracts    {sum(len(v) for v in fut_ok.values()):>10,} bars')

print('\n--- Null % in OHLCV (spot-check first 3 per class) ---')
for label, d in [('Stocks', stock_ok), ('Futures', fut_ok)]:
    for key, df in list(d.items())[:3]:
        cols = [c for c in _OHLCV_COLS[1:] if c in df.columns]
        null_pct = df[cols].isnull().mean().mean() * 100
        date_min = df.index.min().date() if not df.empty else 'n/a'
        date_max = df.index.max().date() if not df.empty else 'n/a'
        print(f'  {label} {key}: {len(df)} rows  {null_pct:.1f}% nulls  [{date_min} -> {date_max}]')

print(f'\nData written to: {DATA_DIR.resolve()}')

In [ ]:
asyncio.run(client.close())
print('HTTP session closed.')